<div style="text-align: center;">

# **Spring 2026 &mdash; CIS 3813<br>Advanced Data Science<br>(Introduction to Machine Learning)**
### Week 8: Naïve Bayes & Support Vector Machines

</div>

**Date:** 23 March 2026  
**Time:** 6:00–9:00 PM  
**Instructor:** Dr. Patrick T. Marsh  
**Course Verse:** "He has shown you, O mortal, what is good. And what does the Lord require of you? To act justly and to love mercy and to walk humbly with your God."  &mdash; *Micah 6:8 (NIV)*

---

## **Week 8 Learning Objectives**

By the end of this lecture, you will be able to:

1. Explain Bayes' theorem and how it underlies the Naïve Bayes classifier
2. Describe what the "naïve" independence assumption means and when it works in practice
3. Apply Gaussian, Multinomial, and Bernoulli Naïve Bayes to appropriate problem types
4. Explain the intuition behind the maximum-margin hyperplane in a Support Vector Machine
5. Describe the "kernel trick" and why it allows SVMs to draw non-linear decision boundaries
6. Use the Week 7 evaluation toolkit to compare Naïve Bayes, SVM, and Logistic Regression
7. Articulate when to prefer each algorithm given a problem's structure and constraints

---

## **Today's Outline**
- Lecture
    1. Review of Last Week & Mastery Assessment
    2. Probabilistic Classification: Thinking Like a Bayesian
    3. Naïve Bayes: Theory
    4. Naïve Bayes: In Practice
    5. Support Vector Machines: The Margin Intuition
    6. The Kernel Trick
    7. SVM Hyperparameters: C and γ
    8. Algorithm Selection: When to Use What
    9. Faith Integration
- Break (10-15 Minutes)
- Lab (or Homework)
- Review
    1. Key Takeaways
    2. Coming Up

---

## **Opening Reflection**

> *"For the Lord gives wisdom; from his mouth come knowledge and understanding."*
> **— Proverbs 2:6 (NIV)**

Today we encounter two classifiers built on radically different philosophies. Naïve Bayes asks: *what does the evidence tell me about the probability of each class?* It updates its beliefs from data using a 250-year-old theorem attributed to an English minister. Support Vector Machines ask instead: *where is the widest street I can draw between the two classes?* It cares nothing about probabilities — only geometry.

Both approaches produce useful predictions, often on the same data. The wisdom lies in knowing which question to ask — and which tool to reach for.

---

## **1.1 Review of Last Week**

**Week 7 key ideas:**
- **The Finley Fallacy:** Accuracy is misleading on imbalanced data — a model predicting the majority class constantly can look accurate while catching nothing
- **The confusion matrix** decomposes predictions into TP, TN, FP, FN — every metric is a ratio built from these four values
- **Precision** = "when I raise the alarm, am I right?" &nbsp;|&nbsp; **Recall** = "did I catch everything?"
- Precision and recall **pull against each other** — the threshold is a business/ethics decision, not a statistical one
- **F1** is the harmonic mean of P and R; **MCC** is the most honest single-number summary for imbalanced data
- **ROC-AUC** measures ranking ability across all thresholds; **PR-AUC** is preferred on highly imbalanced data
- **Class weighting** shifts the decision boundary toward the minority class without modifying the data

**The big question we left open:** *We know how to evaluate classifiers — now let's meet two more that work very differently from logistic regression.*

In [ ]:
# Setup — run this cell first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap

from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.datasets import make_classification, make_moons, make_circles, fetch_20newsgroups
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

np.random.seed(42)
print("Libraries loaded successfully.")

---

## **1.2 Probabilistic Classification: Thinking Like a Bayesian**

### **1.2.1 Two Philosophical Camps**

Every classifier we've seen so far — logistic regression, decision stumps, linear models — answers the question *"which class?"* by drawing a decision boundary. Given features **x**, is this point on the left or right of the line?

Naïve Bayes takes a completely different approach. It doesn't draw a boundary at all. Instead, it asks: *"Given what I observe, what is the probability that this example belongs to each class?"* The predicted class is simply whichever class has the highest probability.

This probabilistic perspective — updating beliefs based on evidence — dates to Reverend Thomas Bayes (1701–1761), an English minister and statistician who worked out the theorem posthumously published in 1763. It is now one of the most consequential ideas in all of statistics.

### **1.2.2 Bayes' Theorem**

$$P(C \mid \mathbf{x}) = \frac{P(\mathbf{x} \mid C) \cdot P(C)}{P(\mathbf{x})}$$

In words:

$$\underbrace{P(C \mid \mathbf{x})}_{\text{posterior}} = \frac{\overbrace{P(\mathbf{x} \mid C)}^{\text{likelihood}} \cdot \overbrace{P(C)}^{\text{prior}}}{\underbrace{P(\mathbf{x})}_{\text{evidence (normalising constant)}}}$$

- **Prior** $P(C)$: Before seeing any features, how likely is class $C$? (e.g., 1% of emails are spam)
- **Likelihood** $P(\mathbf{x} \mid C)$: How likely is this feature pattern *if* the example truly belongs to class $C$?
- **Evidence** $P(\mathbf{x})$: The total probability of observing this feature pattern across *all* classes — a normalising constant that is the same regardless of which class we evaluate
- **Posterior** $P(C \mid \mathbf{x})$: The updated probability of class $C$ *after* observing the features

Since $P(\mathbf{x})$ is the same for every class, classification only requires:

$$\hat{C} = \underset{C}{\arg\max} \; P(\mathbf{x} \mid C) \cdot P(C)$$

The denominator cancels out. We only need the numerator — likelihood times prior — for each class.

#### **1.2.2.1 Why each term matters: an intuitive breakdown**

**The Prior — what you believe before looking at the data**

The prior $P(C)$ encodes your baseline expectation *before* you observe any features. It comes entirely from the training distribution — how often does each class actually appear?

This is more important than it might seem. Suppose you build a model to detect a rare disease that affects 1 in 10,000 people. Even before you look at a single symptom, a very good starting guess is "this person is healthy" — and the prior forces your model to respect that. A classifier that ignores the prior will hallucinate disease everywhere.

**The Likelihood — the generative story**

The likelihood $P(\mathbf{x} \mid C)$ asks: *"If I already knew the class, how probable is this particular combination of feature values?"* This is the reverse of what we actually want (we want to go from features to class), but it's what we can estimate directly from training data. Bayes' theorem is the mathematical "flip" that lets us reverse the conditioning.

**The Evidence — a normalising constant, not the probability of disease**

The denominator $P(\mathbf{x})$ is the total probability of observing this feature pattern across *all* classes:

$$P(\mathbf{x}) = \sum_{c} P(\mathbf{x} \mid C=c) \cdot P(C=c)$$

A common misconception: in the medical example below, $P(+)$ is **not** the probability of having the disease — it is the probability of **testing positive**, which includes both true positives (sick people who correctly test positive) and false positives (healthy people who incorrectly test positive). It ensures all posteriors sum to 1, making them proper probabilities. Since this value is the same regardless of which class we're evaluating, it plays no role in which class wins — which is why we can drop it for classification.

**The Posterior — your updated belief**

The posterior $P(C \mid \mathbf{x})$ is the answer to the question we actually care about: given everything I've observed, which class is most likely? It combines your prior belief with the evidence from the data. The Bayesian view of learning is precisely this: classification is the act of updating priors with likelihoods.

#### **1.2.2.2 A medical diagnosis walkthrough**

Medical diagnosis is the canonical illustration of Bayes' theorem because the stakes make the math feel real.

Suppose a disease affects 1% of the population. A test for the disease is 95% sensitive (catches 95% of true cases) and 90% specific (correctly rules out 90% of healthy people). You test positive. What is the probability you actually have the disease?

Most people's gut answer is around 90–95%. The correct answer is roughly **8.7%** — and understanding why is the heart of Bayes' theorem.

Let $D$ = "has disease", $+$ = "tests positive".

$$P(D \mid +) = \frac{P(+ \mid D) \cdot P(D)}{P(+)}$$

- $P(D) = 0.01$ — the prior: 1% of people have the disease
- $P(+ \mid D) = 0.95$ — sensitivity: 95% of sick people test positive
- $P(+ \mid \neg D) = 0.10$ — false positive rate: 10% of healthy people also test positive

The denominator $P(+)$ is the total probability of a positive test result — **not** the probability of disease:
$$P(+) = P(+ \mid D) \cdot P(D) + P(+ \mid \neg D) \cdot P(\neg D) = 0.95 \times 0.01 + 0.10 \times 0.99 = 0.1085$$

Therefore:
$$P(D \mid +) = \frac{0.95 \times 0.01}{0.1085} \approx 0.087$$

Why so low? Because the disease is *rare* — the prior is 1%. Even a fairly good test generates far more false positives (from the 99% healthy population) than true positives (from the 1% sick population). The prior dominates.

> **Key insight:** This result is not a flaw in the theorem — it is the theorem working correctly. It is why rare-disease screening programs almost always require a confirmatory test before treatment. A single positive from a population screen is weak evidence because the base rate is low. This is the "base rate fallacy" that Bayes' theorem helps us avoid.

**What changes with a stronger prior?** If we test only symptomatic patients where prevalence is 20% (not 1%), the same test now yields $P(D \mid +) \approx 70\%$ — because the prior is much more favorable. Same test, radically different posterior. The prior matters enormously.

#### **1.2.2.3 Sequential updating: Bayes as a learning process**

One of the most elegant properties of Bayes' theorem is that it is *composable*: the posterior from one observation becomes the prior for the next.

Suppose our spam filter sees the word "FREE" in an email. We update our belief from the prior (say, 20% spam) to a new posterior (75% spam, as computed in the code below). Now suppose the same email also contains the word "CLICK". We use our updated 75% as the new prior and apply Bayes again:

$$P(\text{spam} \mid \text{FREE, CLICK}) \propto P(\text{CLICK} \mid \text{spam}) \times 0.75$$

Each new piece of evidence sharpens our estimate. This sequential view is exactly how Naïve Bayes processes multiple features — it applies the theorem once per feature (under the independence assumption), multiplying likelihoods together. The math and the intuition are the same.

> **This is what learning looks like from a Bayesian perspective:** you start with a prior, accumulate evidence, and your beliefs converge toward the truth as more data arrives. The prior matters less and less as evidence accumulates — a good epistemic model for scientific reasoning as well as machine learning.

#### **1.2.2.4 The log trick in practice**

With many features, the product of many small probabilities becomes numerically unstable — floating-point underflow sends the value to zero before you can compare classes. The solution is to take logarithms, converting the product into a sum:

$$\log P(C \mid \mathbf{x}) \propto \log P(C) + \sum_{i=1}^{n} \log P(x_i \mid C)$$

Since $\log$ is a monotone increasing function, the class with the highest log-posterior is still the same class with the highest posterior. Sklearn's Naïve Bayes implementations use log-probabilities internally for exactly this reason — the `feature_log_prob_` attribute you'll inspect in the lab stores these values directly.


In [ ]:
# ── A concrete Bayes' theorem example: email spam detection ──────────────
#
# Suppose:
#   - 20% of incoming emails are spam:  P(Spam) = 0.20
#   - The word "FREE" appears in 60% of spam emails: P(FREE | Spam) = 0.60
#   - The word "FREE" appears in 5% of legitimate emails: P(FREE | Ham) = 0.05
# Question: if an email contains "FREE", what is the probability it is spam?

P_spam = 0.20
P_ham  = 1 - P_spam

P_FREE_given_spam = 0.60
P_FREE_given_ham  = 0.05

# Law of Total Probability: P(FREE) = P(FREE|Spam)P(Spam) + P(FREE|Ham)P(Ham)
P_FREE = P_FREE_given_spam * P_spam + P_FREE_given_ham * P_ham

# Bayes' theorem
P_spam_given_FREE = (P_FREE_given_spam * P_spam) / P_FREE
P_ham_given_FREE  = (P_FREE_given_ham  * P_ham)  / P_FREE

print("═" * 55)
print("  Spam Detection with Bayes' Theorem")
print("═" * 55)
print(f"  Prior P(Spam)               : {P_spam:.2f}")
print(f"  Prior P(Ham)                : {P_ham:.2f}")
print(f"  P(\"FREE\" | Spam)             : {P_FREE_given_spam:.2f}")
print(f"  P(\"FREE\" | Ham)              : {P_FREE_given_ham:.2f}")
print(f"  P(\"FREE\") overall            : {P_FREE:.4f}")
print()
print(f"  P(Spam | \"FREE\")  = {P_spam_given_FREE:.4f}  ({P_spam_given_FREE:.1%})")
print(f"  P(Ham  | \"FREE\")  = {P_ham_given_FREE:.4f}  ({P_ham_given_FREE:.1%})")
print()
print("  Prediction: SPAM (highest posterior probability)")
print()
print("  Note: Seeing 'FREE' updated our belief from 20% to 75%!")
print("  That's Bayes' theorem at work — evidence updating prior belief.")

---

## **1.3 Naïve Bayes: Theory**

### **1.3.1 The Naïve Assumption**

In practice, emails (and most real-world examples) have many features — thousands of words. Computing the joint likelihood $P(w_1, w_2, \ldots, w_n \mid C)$ for all word combinations is **computationally impossible**: there are too many combinations to estimate reliably from data.

To see why, consider a vocabulary of just 10,000 words. The number of possible two-word co-occurrences is already $10{,}000^2 = 100{,}000{,}000$. For three-word combinations it's $10{,}000^3 = 10^{12}$. Even with a very large training set, most of these combinations would never appear, making their probabilities impossible to estimate. This is called the **curse of dimensionality** — the joint probability space grows exponentially with the number of features.

Naïve Bayes makes one bold simplification: **assume each feature is conditionally independent of every other feature, given the class**.

$$P(\mathbf{x} \mid C) = P(x_1 \mid C) \cdot P(x_2 \mid C) \cdots P(x_n \mid C) = \prod_{i=1}^{n} P(x_i \mid C)$$

This collapses the problem dramatically. Instead of estimating one enormous joint distribution, we only need to estimate $n$ small distributions — one per feature per class. For 10,000 words and 2 classes, that's just 20,000 values instead of a number with 40,000 zeros.

This is the "naïve" part — the features are almost *never* truly independent. The words "New" and "York" in a document are clearly not independent. The words "free" and "prize" in a spam email co-occur far more often than chance would predict. Yet despite this incorrect assumption, the classifier works remarkably well in practice. Why?

Because we don't need the probability estimates to be perfectly calibrated — we only need the **ranking** to be correct. Even if all the posteriors are wrong by the same systematic factor, the class with the highest posterior is still the same class. The naïve assumption introduces bias into the probability estimates, but often does very little damage to the final classification decision.

There is also a practical reason it works: in high-dimensional spaces, the signal from each individual feature is relatively weak, and the independence assumption distributes that signal across features without dramatically distorting the relative ordering of class probabilities. It's a crude approximation that happens to be "good enough" surprisingly often.

> **Technical note:** The naïve independence assumption makes Naïve Bayes a *generative* model — it models how each class generates features. Logistic regression, by contrast, is a *discriminative* model — it models the boundary between classes directly. Generative models have useful properties: they can handle missing features gracefully (just omit the missing term in the product), they can be updated with new data without retraining from scratch, and they tend to be more data-efficient when training data is scarce. Discriminative models generally produce better-calibrated probability estimates when data is abundant.

### **1.3.2 Three Variants of Naïve Bayes**

The choice of variant depends on the nature of your features — specifically, what probability distribution you assume each feature follows within a class:

| Variant | Feature Type | Likelihood Model | Typical Use Case |
|---------|-------------|------------------|------------------|
| **Gaussian NB** | Continuous | $P(x_i \mid C) = \mathcal{N}(\mu_{iC}, \sigma_{iC}^2)$ | Numeric features (height, weight, sensor readings) |
| **Multinomial NB** | Count / Frequency | Multinomial distribution over counts | Word counts, TF-IDF in text classification |
| **Bernoulli NB** | Binary (0/1) | Bernoulli distribution | Word presence/absence, binary features |

**Gaussian NB** assumes that within each class, each continuous feature follows a normal distribution. During training, it estimates the mean $\mu_{iC}$ and variance $\sigma_{iC}^2$ for each feature $i$ and class $C$ — just two numbers per feature per class. At prediction time, it evaluates how likely a new value is under that Gaussian curve. This is fast, simple, and works well when the data is roughly bell-shaped within each class. It can struggle when features are heavily skewed or multimodal.

**Multinomial NB** is the standard choice for text classification with word counts. It models the likelihood of observing a particular word frequency vector given a class, and is mathematically equivalent to counting how often each word appears across all documents of each class. The key parameter is the word frequency relative to all words in that class — words that appear disproportionately often in spam (relative to ham) get high spam likelihood scores.

**Bernoulli NB** uses only whether each word is present or absent — it ignores how many times a word appears. This can be appropriate for very short texts where a word is unlikely to appear more than once anyway, or when repeated words don't carry additional signal. It also explicitly penalises the *absence* of words that are characteristic of the non-target class, which Multinomial NB does not do.

### **1.3.3 Laplace Smoothing**

There is a practical problem that arises immediately: what happens when a word in the test set was never seen during training for a particular class? Its count is zero, so its estimated probability $P(x_i \mid C) = 0$, and since we're multiplying probabilities together, the entire posterior for that class collapses to zero — regardless of all other evidence. One unseen word poisons the whole calculation.

**Laplace smoothing** (also called additive smoothing) prevents this by adding a small constant $\alpha$ (typically 1) to every word's count before computing probabilities:

$$P(x_i \mid C) = \frac{\text{count}(x_i, C) + \alpha}{\sum_j \text{count}(x_j, C) + \alpha \cdot |V|}$$

where $|V|$ is the vocabulary size. With $\alpha = 1$, every word is treated as if it appeared at least once in every class. No word can ever have zero probability.

The intuition is simple: we're saying "I haven't seen this word in spam yet, but I can't be certain it will never appear. I'll assign it a very small probability rather than zero." The larger $\alpha$ is, the more aggressively we smooth toward a uniform distribution — essentially discounting the training data in favour of the uniform prior. In practice, $\alpha = 1$ (the default in sklearn) works well for most text classification tasks, though it can be tuned like any hyperparameter.

> **Why this matters beyond text:** Laplace smoothing is an instance of a general principle in statistics called **regularisation toward a prior**. With $\alpha = 1$, we are implicitly treating each count as if it started with one pseudo-observation — a very mild Bayesian prior that says "all words are equally likely before I see any data." This connects Naïve Bayes smoothing directly to the Bayesian framework the whole model is built on.

### **1.3.4 Log-Probabilities in Practice**

With many features, multiplying hundreds or thousands of small probabilities together produces numbers too small for floating-point arithmetic to represent — they underflow to zero before you can compare classes. The solution is to work in log-space, converting the product into a sum:

$$\log P(C \mid \mathbf{x}) \propto \log P(C) + \sum_{i=1}^{n} \log P(x_i \mid C)$$

Since $\log$ is a monotone increasing function, the class with the highest log-posterior is the same as the class with the highest posterior. The ranking is preserved, so classification still works correctly.

This is not just a numerical trick — it also makes the math simpler to reason about. Each feature contributes an *additive* piece of evidence in log-space. A feature strongly associated with class $C$ contributes a large (less negative) log-likelihood; a feature strongly associated with the other class contributes a very negative one. You can literally read off the evidence for and against each class by looking at the individual terms of the sum. This is why sklearn's `feature_log_prob_` attribute is so useful for interpreting a trained Naïve Bayes model — each value directly represents how much log-evidence that feature contributes toward its class.


In [ ]:
# ── Gaussian Naïve Bayes: manual derivation on a tiny dataset ────────────
# Suppose we have two features (sepal-length-like) and two classes.
# We'll compute the Gaussian NB prediction from scratch, then verify with sklearn.

# Toy training data: class 0 clusters around (2, 2); class 1 around (6, 6)
X_toy = np.array([
    [1.5, 2.0], [2.0, 1.8], [2.5, 2.2], [1.8, 2.5],   # class 0
    [5.5, 6.0], [6.0, 5.8], [6.5, 6.2], [5.8, 6.5],   # class 1
])
y_toy = np.array([0, 0, 0, 0, 1, 1, 1, 1])

# Test point
x_new = np.array([4.0, 4.5])

# ── Manual Gaussian NB ───────────────────────────────────────────────────
from scipy.stats import norm

for cls in [0, 1]:
    subset = X_toy[y_toy == cls]
    prior  = np.mean(y_toy == cls)                        # P(C)
    means  = subset.mean(axis=0)                          # μ per feature
    stds   = subset.std(axis=0)                           # σ per feature
    log_likelihood = np.sum(norm.logpdf(x_new, means, stds))  # Σ log P(xi|C)
    log_posterior  = np.log(prior) + log_likelihood
    print(f"Class {cls}:  prior={prior:.2f}  mean={means}  std={stds.round(2)}")
    print(f"         log-likelihood={log_likelihood:.4f}  log-posterior={log_posterior:.4f}")

# ── Verify with sklearn ──────────────────────────────────────────────────
gnb = GaussianNB()
gnb.fit(X_toy, y_toy)
pred  = gnb.predict([x_new])[0]
proba = gnb.predict_proba([x_new])[0]
print()
print(f"sklearn GaussianNB prediction: class {pred}")
print(f"  P(class 0 | x) = {proba[0]:.4f}")
print(f"  P(class 1 | x) = {proba[1]:.4f}")
print(f"  (x_new = {x_new} sits between the two clusters — uncertain prediction)")

---

## **1.4 Naïve Bayes: In Practice**

### **1.4.1 Text Classification — Where Naïve Bayes Shines**

Naïve Bayes is the workhorse of text classification, and has been since the earliest days of spam filtering in the late 1990s. Paul Graham's influential 2002 essay "A Plan for Spam" described a Bayesian filter that outperformed every previous approach — and the core idea was exactly what we've been studying. Understanding *why* Naïve Bayes is so well-suited to text helps build intuition for when to use it elsewhere.

**Why text is an ideal domain for Naïve Bayes:**

1. **Speed:** Training is a single pass through the data — just compute word counts per class. There is no iterative optimization, no gradient descent, no loss function to minimize. For a dataset with $n$ documents and $|V|$ vocabulary words, training is $O(n \cdot |V|)$ — as fast as reading the data. This matters when vocabularies are large (100k+ words) or data is streaming in continuously.

2. **High dimensionality is natural:** Text vectors are extremely high-dimensional (one dimension per word) but very sparse (most documents use a tiny fraction of the vocabulary). Many algorithms struggle with this — distance metrics become meaningless, matrices become impossible to invert. Naïve Bayes is immune because it never computes distances or matrix operations. Each feature is handled independently.

3. **Data efficiency:** The naïve independence assumption dramatically reduces the number of parameters to estimate. For Multinomial NB with a 10,000-word vocabulary and 2 classes, you only need to estimate 20,000 word probabilities — versus astronomical numbers for a full joint model. This means Naïve Bayes can produce reasonable classifiers with hundreds of training examples, not millions.

4. **Interpretability:** Because the model stores a log-probability for every word in every class, you can directly answer the question "which words drive this prediction?" — simply look at the word log-probabilities for the predicted class. No saliency maps, no SHAP values, no approximations. This is especially valuable in regulated industries where model decisions need to be explained to non-technical stakeholders.

5. **Surprisingly competitive accuracy:** On text tasks, Naïve Bayes is often within a few percentage points of logistic regression and linear SVMs, despite being an order of magnitude simpler. The independence assumption is violated — adjacent words clearly co-occur non-randomly — yet the model works well anyway. This is sometimes called the "Naïve Bayes paradox" and is an active area of theoretical research.

**When to use Multinomial vs. Bernoulli NB for text:**

- Use **Multinomial NB** when word frequency matters. In a spam email, "FREE" appearing five times is stronger evidence than appearing once. Multinomial NB captures this because its likelihood is proportional to word counts.
- Use **Bernoulli NB** when you care only about which words appear, not how often. It explicitly models the probability that a word is *absent* from a document, which Multinomial NB does not — this can help in domains where certain words are strong negative indicators.
- As a practical heuristic: Multinomial NB usually wins on longer documents (news articles, emails, reviews); Bernoulli NB can be competitive on very short texts (tweets, titles).

### **1.4.2 Why Probability Calibration Suffers**

There is an important practical caveat: while Naïve Bayes produces *rankings* well (which class is more likely), the actual probability *values* it outputs are often poorly calibrated — sometimes absurdly so. A model might output $P(\text{spam}) = 0.9999$ when the true probability is closer to 0.75.

The reason is the independence assumption. When two features are positively correlated — say, "free" and "prize" both appear together in spam — the model double-counts their evidence. It effectively sees two independent votes for spam, when in reality they carry less combined information than two truly independent signals would. This causes the model to be overconfident: the posterior gets pushed toward 0 or 1 more aggressively than the data warrants.

**Practical implication:** If you need reliable probability estimates (e.g., for risk scoring, decision thresholds, or combining with other models), Naïve Bayes needs to be calibrated post-hoc using techniques like Platt scaling or isotonic regression. If you only need rankings or binary predictions, this limitation rarely matters.

### **1.4.3 Naïve Bayes: Strengths and Weaknesses**

| Strength | Limitation |
|----------|------------|
| Extremely fast to train — single pass through data | Independence assumption violated in almost all real data |
| Works well with very small datasets | Probability estimates often poorly calibrated (overconfident) |
| Handles high-dimensional sparse feature spaces naturally | Correlated features get double-counted, inflating confidence |
| Naturally handles multi-class problems without modification | Gaussian NB assumes normally distributed features within each class |
| New classes can be added without full retraining | Not suitable when feature interactions are important |
| Highly interpretable — feature log-probs directly readable | Discrete NB variants require non-negative features |
| Missing features handled gracefully — just omit the term | Outperformed by discriminative models when data is abundant |


> **Rule of thumb:** Naïve Bayes is almost always worth trying first on text classification. It trains in milliseconds, is easily interpretable, and sets a surprisingly strong baseline. If it performs within a few points of a tuned logistic regression or SVM, the simplicity advantage often makes it the right production choice. When features are strongly correlated (structured tabular data) or interactions matter (e.g., "not good" means something different than "not" and "good" separately), prefer logistic regression or tree-based models.


In [ ]:
# ── Text classification: 20 Newsgroups (4 categories) ───────────────────
# A classic benchmark: news posts from four discussion groups.
# We compare Multinomial NB (word counts) vs. Bernoulli NB (word presence).

categories = ['rec.sport.baseball', 'sci.space', 'talk.politics.guns', 'comp.graphics']

newsgroups_train = fetch_20newsgroups(subset='train', categories=categories,
                                      remove=('headers', 'footers', 'quotes'),
                                      random_state=42)
newsgroups_test  = fetch_20newsgroups(subset='test',  categories=categories,
                                      remove=('headers', 'footers', 'quotes'),
                                      random_state=42)

print(f"Training examples : {len(newsgroups_train.data)}")
print(f"Test examples     : {len(newsgroups_test.data)}")
print(f"Categories        : {newsgroups_train.target_names}")
print()

# --- Sample document ---
print("── Sample document (first 300 chars) ──")
print(newsgroups_train.data[0][:300])
print(f"\nLabel: {newsgroups_train.target_names[newsgroups_train.target[0]]}")

In [ ]:
# ── Multinomial NB with CountVectorizer ──────────────────────────────────
mnb_pipeline = Pipeline([
    ('vectorizer', CountVectorizer(stop_words='english', max_features=20000)),
    ('classifier', MultinomialNB(alpha=1.0))  # alpha = Laplace smoothing
])

# ── Bernoulli NB with binary features ───────────────────────────────────
bnb_pipeline = Pipeline([
    ('vectorizer', CountVectorizer(stop_words='english', max_features=20000,
                                   binary=True)),   # word presence only
    ('classifier', BernoulliNB(alpha=1.0))
])

# ── Logistic Regression baseline ─────────────────────────────────────────
lr_pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer(stop_words='english', max_features=20000)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

models = {
    'Multinomial NB': mnb_pipeline,
    'Bernoulli NB':   bnb_pipeline,
    'Logistic Regression (baseline)': lr_pipeline,
}

print(f"{'Model':<35} {'Accuracy':>10} {'Macro F1':>10}")
print("─" * 58)
results = {}
for name, pipe in models.items():
    pipe.fit(newsgroups_train.data, newsgroups_train.target)
    preds = pipe.predict(newsgroups_test.data)
    acc = accuracy_score(newsgroups_test.target, preds)
    f1  = f1_score(newsgroups_test.target, preds, average='macro')
    print(f"  {name:<33} {acc:>10.4f} {f1:>10.4f}")
    results[name] = {'acc': acc, 'f1': f1, 'preds': preds}

In [ ]:
# ── Inspect the most informative features per class (Multinomial NB) ─────
# This is one of NB's great advantages: interpretability.

vectorizer = mnb_pipeline.named_steps['vectorizer']
nb_model   = mnb_pipeline.named_steps['classifier']
vocab      = vectorizer.get_feature_names_out()

print("Top 12 most informative words per category (Multinomial NB)")
print("═" * 65)
for i, cat in enumerate(newsgroups_train.target_names):
    # Log probability of each word given this class
    log_probs = nb_model.feature_log_prob_[i]
    top_idx   = np.argsort(log_probs)[-12:][::-1]
    top_words = ', '.join(vocab[top_idx])
    print(f"\n  {cat}:")
    print(f"  {top_words}")

In [ ]:
# ── Confusion matrix: Multinomial NB ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, res) in zip(axes, list(results.items())[:2]):
    cm = confusion_matrix(newsgroups_test.target, res['preds'])
    disp = ConfusionMatrixDisplay(cm, display_labels=[
        'baseball', 'space', 'guns', 'graphics'
    ])
    disp.plot(ax=ax, colorbar=False, cmap='Blues', xticks_rotation=30)
    ax.set_title(f"{name}\nAccuracy={res['acc']:.3f}  Macro F1={res['f1']:.3f}",
                 fontsize=10)

plt.suptitle('Text Classification — 20 Newsgroups (4 categories)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey observation: Naïve Bayes (especially Multinomial) is competitive")
print("with Logistic Regression on text — and trains in a fraction of the time.")
print("Which categories are most often confused? Does that make intuitive sense?")

---

## **1.5 Support Vector Machines: The Margin Intuition**

### **1.5.1 Why a New Algorithm?**

We now shift from probabilistic thinking to geometric thinking. Naïve Bayes asked: *"What is the probability that this point belongs to each class?"* Support Vector Machines ask a completely different question: *"Where is the widest possible street I can draw between the two classes?"*

This is not just a stylistic difference — it reflects a fundamentally different philosophy about what makes a classifier good. Naïve Bayes worries about every single training point, assigning each one a probability. SVMs deliberately ignore most of the training data and focus entirely on the handful of points that are hardest to classify — the ones right at the boundary.

The intuition for why this works: if you have a clear separation between classes, the points far from the boundary are easy — almost any reasonable classifier would get them right. What distinguishes a *good* boundary from a *great* boundary is how well it handles the ambiguous region in the middle. SVMs optimize exclusively for that.

### **1.5.2 From Logistic Regression to SVMs**

Logistic regression finds a hyperplane by minimizing log-loss across *all* training examples — every point contributes to the gradient update, weighted by how wrong the model is about it. Points far from the boundary contribute a tiny gradient; points near the boundary contribute a large one. But all points contribute *something*.

SVMs take a more extreme stance: **once a point is correctly classified and sufficiently far from the boundary, it contributes nothing at all.** Only the points that are closest to the boundary — the support vectors — determine where the boundary goes. Remove any other training point and the boundary stays exactly the same.

This produces a classifier that is robust to irrelevant data. If someone adds 1,000 correctly-classified points far from the boundary to your training set, logistic regression's boundary will shift slightly. An SVM's boundary will not move at all — those points have no leverage.

### **1.5.3 The Maximum-Margin Hyperplane**

Given a linearly separable dataset, there are infinitely many hyperplanes that correctly classify every training point. Which one should we choose?

Consider the problem geometrically. Any separating hyperplane has some distance to the nearest positive example and some distance to the nearest negative example. The **margin** is the sum of these two distances — it is the width of the empty "street" between the classes. SVMs choose the hyperplane that **maximizes this margin**.

Why does maximizing the margin help? The intuition comes from generalization theory. A wider margin means:
- **More tolerance for noise:** If a training point is slightly mislabeled or measured with error, a wide-margin classifier is less likely to be fooled by it.
- **Better generalization bounds:** The VC-dimension theory behind SVMs (developed by Vapnik and Chervonenkis) shows that classifiers with larger margins have lower complexity, and therefore smaller bounds on out-of-sample error. In other words, maximizing the margin is a principled way to reduce overfitting.
- **A unique, stable solution:** Among all infinitely many separating hyperplanes, the maximum-margin one is the unique choice that is maximally "between" the classes. It's geometrically natural in a way that other choices are not.

The decision boundary is:
$$\mathbf{w}^\top \mathbf{x} + b = 0$$

The two **margin boundaries** (one per class) are:
$$\mathbf{w}^\top \mathbf{x} + b = +1 \quad \text{(positive class boundary)}$$
$$\mathbf{w}^\top \mathbf{x} + b = -1 \quad \text{(negative class boundary)}$$

The margin width is $\frac{2}{\|\mathbf{w}\|}$. Maximizing the margin is therefore equivalent to minimizing $\|\mathbf{w}\|$ — or equivalently, minimizing $\frac{1}{2}\|\mathbf{w}\|^2$ (the squared form is mathematically convenient for optimization).

**Support vectors** are the training points that lie exactly on the margin boundaries. They are the only examples that matter for determining the decision boundary. Everything else is irrelevant once training is done — which is also why SVMs can be memory-efficient at inference time: you only need to store the support vectors, not the full training set.

### **1.5.4 Soft-Margin SVM: The C Hyperparameter**

Real data is rarely perfectly linearly separable. Two things cause this:
1. **Genuine overlap:** The classes may not be separable at all — some spam emails use very normal-looking language, and some legitimate emails look spammy.
2. **Noise and outliers:** A single misplaced or mislabeled point can make an otherwise separable dataset technically non-separable.

The **soft-margin SVM** handles this by allowing some points to violate the margin — or even cross the decision boundary — by introducing **slack variables** $\xi_i \geq 0$. Each slack variable measures how much a point violates its margin constraint: $\xi_i = 0$ means the point is on the correct side of its margin boundary; $\xi_i = 1$ means it is exactly on the decision boundary; $\xi_i > 1$ means it is misclassified.

The optimization problem becomes:
$$\text{Minimize: } \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^{n} \xi_i$$

The first term maximizes the margin; the second term penalizes violations. The hyperparameter **C** is the exchange rate between these two objectives:

- **Large C:** The penalty for violations is high → the model tries hard to classify every point correctly → the margin shrinks to accommodate difficult points → higher risk of overfitting to noise.
- **Small C:** Violations are cheap → the model accepts more misclassifications in exchange for a wider margin → better generalization when data is noisy.

> **Don't confuse C with regularization strength.** In Ridge/Lasso regression, a *larger* regularization parameter ($\lambda$) means *more* regularization. In SVMs, a *larger* C means *less* regularization. They are inverses: $C \approx \frac{1}{\lambda}$. sklearn's `LogisticRegression` also uses a `C` parameter with the same convention as SVMs — larger C means less regularization.

A useful way to think about C: it controls how much you trust your training data. If your data is clean and well-labeled, use a larger C — you can afford to classify everything correctly. If your data is noisy or you expect distribution shift at test time, use a smaller C — accept some training errors in exchange for a more robust boundary.


<!-- Section 1.5 content moved to previous cell -->


In [ ]:
# ── Visualize the SVM margin and support vectors ─────────────────────────
# Use a simple 2D linearly separable dataset so we can see everything clearly.

from sklearn.datasets import make_blobs

X_sv, y_sv = make_blobs(n_samples=80, centers=2, cluster_std=1.0,
                         random_state=7)

# Hard-margin SVM (large C → near hard margin)
svm_linear = SVC(kernel='linear', C=1000, random_state=42)
svm_linear.fit(X_sv, y_sv)

# ── Plot setup ────────────────────────────────────────────────────────────
def plot_svm(ax, model, X, y, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, levels=[-1e5, 0, 1e5],
                colors=['#d4e6f1', '#fadbd8'], alpha=0.5)
    ax.contour(xx, yy, Z, levels=[-1, 0, 1],
               linestyles=['--', '-', '--'],
               colors=['steelblue', 'black', 'tomato'], linewidths=[1.5, 2.5, 1.5])

    ax.scatter(X[y == 0, 0], X[y == 0, 1], c='steelblue', s=50,
               edgecolors='white', linewidths=0.5, label='Class 0')
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c='tomato', s=50,
               edgecolors='white', linewidths=0.5, label='Class 1')

    # Highlight support vectors
    sv = model.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1], s=180, edgecolors='gold',
               facecolors='none', linewidths=2.0, label=f'Support vectors ({len(sv)})')

    margin = 2 / np.linalg.norm(model.coef_) if hasattr(model, 'coef_') else None
    if margin:
        ax.set_title(f"{title}\nMargin = {margin:.3f} | Support vectors = {len(sv)}",
                     fontsize=10)
    else:
        ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')

fig, ax = plt.subplots(figsize=(7, 6))
plot_svm(ax, svm_linear, X_sv, y_sv, 'Linear SVM — Maximum Margin Hyperplane')
plt.tight_layout()
plt.show()

print("The solid line is the decision boundary (w·x + b = 0).")
print("The dashed lines are the margin boundaries (w·x + b = ±1).")
print("Gold circles are support vectors — the only points that matter.")

---

## **1.6 The Kernel Trick**

### **1.6.1 The Problem with Linear Boundaries**

Linear SVMs work beautifully when classes are (approximately) linearly separable. But many real datasets are not. Consider two classic examples:
- **Concentric circles:** Points inside a circle belong to class A; points outside belong to class B. No straight line separates them.
- **XOR pattern:** Class A occupies the top-left and bottom-right quadrants; class B occupies the other two. Again, no straight line works.

The obvious fix is to engineer new features that make the problem linearly separable. For the circles example, if you add a feature $r = x_1^2 + x_2^2$ (the squared distance from the origin), the two classes are perfectly separated by a horizontal cut in the $(x_1, x_2, r)$ space. The boundary is linear in the expanded space, even though it looks curved in the original space.

This works — but it gets expensive fast. Adding all polynomial features of degree $d$ over $p$ original features creates $\binom{p+d}{d}$ new features. For $p = 100$ features and degree $d = 3$, that's over 176,000 new dimensions. The computation becomes prohibitive.

### **1.6.2 The Key Insight: SVMs Only Need Dot Products**

Here is the crucial observation that makes the kernel trick possible: when you write out the SVM optimization problem in its **dual form**, the training data appears only as **dot products** between pairs of points — never as individual coordinates.

The dual objective depends on $\mathbf{x}_i^\top \mathbf{x}_j$ for all pairs $(i, j)$ — not on $\mathbf{x}_i$ or $\mathbf{x}_j$ individually. The same is true at prediction time: classifying a new point $\mathbf{x}$ requires computing $\mathbf{x}^\top \mathbf{x}_i$ for each support vector $\mathbf{x}_i$.

This means: if we want to work in an expanded feature space $\phi(\mathbf{x})$, we never actually need to *compute* $\phi(\mathbf{x})$. We only need to compute $\phi(\mathbf{x}_i)^\top \phi(\mathbf{x}_j)$ — the dot product *between* transformed points. A **kernel function** does exactly this:

$$K(\mathbf{x}_i, \mathbf{x}_j) = \phi(\mathbf{x}_i)^\top \phi(\mathbf{x}_j)$$

We evaluate $K$ directly in the original space — a cheap computation — and get the same result as if we had mapped both points to the expanded space and taken their dot product there. We never construct $\phi(\mathbf{x})$ explicitly.

### **1.6.3 A Concrete Example: The Polynomial Kernel**

Consider two 2D points $\mathbf{x} = (x_1, x_2)$ and $\mathbf{z} = (z_1, z_2)$. The degree-2 polynomial kernel is:

$$K(\mathbf{x}, \mathbf{z}) = (\mathbf{x}^\top \mathbf{z})^2 = (x_1 z_1 + x_2 z_2)^2$$

Expanding this:
$$= x_1^2 z_1^2 + 2 x_1 x_2 z_1 z_2 + x_2^2 z_2^2$$

This is exactly the dot product of the transformed vectors $\phi(\mathbf{x}) = (x_1^2, \sqrt{2}x_1 x_2, x_2^2)$ and $\phi(\mathbf{z}) = (z_1^2, \sqrt{2}z_1 z_2, z_2^2)$. Computing $K$ requires one dot product and one squaring — two operations. Computing $\phi(\mathbf{x})^\top \phi(\mathbf{z})$ explicitly requires constructing the 3D vectors first. For high dimensions and high degrees, this gap becomes enormous.

### **1.6.4 The RBF Kernel: An Infinite-Dimensional Mapping**

The most widely used kernel — the Radial Basis Function (RBF), also called the Gaussian kernel — is even more remarkable:

$$K(\mathbf{x}_i, \mathbf{x}_j) = \exp\left(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2\right)$$

The corresponding feature map $\phi$ maps each point into an *infinite-dimensional* space. This is not a metaphor — the Taylor series expansion of the exponential function produces an infinite sum of polynomial terms of all degrees. In principle, the RBF-SVM can represent arbitrarily complex decision boundaries.

The kernel value has a beautiful interpretation: it is a **similarity measure**. Two identical points have $\|\mathbf{x}_i - \mathbf{x}_j\|^2 = 0$, so $K = e^0 = 1$ (maximum similarity). As points move apart, $K$ decays toward 0. The parameter $\gamma$ controls how quickly this decay happens:

- **Large $\gamma$:** Similarity decays quickly with distance → each training point influences only a small neighborhood around itself → the decision boundary can form tight, complex shapes → **overfitting risk**.
- **Small $\gamma$:** Similarity decays slowly → each training point influences a large region → the boundary is smoother and more global → **underfitting risk**.

You can think of $\gamma$ as controlling the "reach" of each training point. A large $\gamma$ means each support vector only "votes" for its immediate neighborhood. A small $\gamma$ means each support vector influences predictions far away.

### **1.6.5 Common Kernels**

| Kernel | Formula | Implicit Feature Space | Typical Use Case |
|--------|---------|------------------------|------------------|
| **Linear** | $\mathbf{x}_i^\top \mathbf{x}_j$ | Original space | Text, high-dim sparse data |
| **Polynomial** | $(\mathbf{x}_i^\top \mathbf{x}_j + r)^d$ | All polynomials up to degree $d$ | Image features, moderate non-linearity |
| **RBF (Gaussian)** | $\exp(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2)$ | Infinite-dimensional | General-purpose, default choice |
| **Sigmoid** | $\tanh(\gamma \mathbf{x}_i^\top \mathbf{x}_j + r)$ | Related to neural nets | Rarely preferred in practice |

> **Which kernel to choose?** Start with RBF — it is the most flexible and works well across a wide range of problems. Use linear when you have high-dimensional sparse data (text, genomics) where the linear kernel is often already expressive enough and much faster. Use polynomial if you have domain knowledge suggesting polynomial interactions matter.


In [ ]:
# ── Effect of C on decision boundary (noisy, overlapping data) ───────────
X_noisy, y_noisy = make_classification(n_samples=120, n_features=2,
                                        n_redundant=0, n_informative=2,
                                        n_clusters_per_class=1,
                                        class_sep=1.2, random_state=12)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
C_values = [0.01, 1.0, 1000.0]

for ax, C_val in zip(axes, C_values):
    model = SVC(kernel='linear', C=C_val, random_state=42)
    model.fit(X_noisy, y_noisy)
    train_acc = model.score(X_noisy, y_noisy)

    x_min, x_max = X_noisy[:, 0].min() - 0.5, X_noisy[:, 0].max() + 0.5
    y_min, y_max = X_noisy[:, 1].min() - 0.5, X_noisy[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, levels=[-1e5, 0, 1e5],
                colors=['#d4e6f1', '#fadbd8'], alpha=0.5)
    ax.contour(xx, yy, Z, levels=[-1, 0, 1],
               linestyles=['--', '-', '--'],
               colors=['steelblue', 'black', 'tomato'], linewidths=[1.2, 2.0, 1.2])
    ax.scatter(X_noisy[y_noisy == 0, 0], X_noisy[y_noisy == 0, 1],
               c='steelblue', s=40, edgecolors='white', linewidths=0.5)
    ax.scatter(X_noisy[y_noisy == 1, 0], X_noisy[y_noisy == 1, 1],
               c='tomato', s=40, edgecolors='white', linewidths=0.5)
    sv = model.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1], s=150, edgecolors='gold',
               facecolors='none', linewidths=1.8)

    margin = 2 / np.linalg.norm(model.coef_)
    ax.set_title(f"C = {C_val}\nTrain acc={train_acc:.2f} | "
                 f"Margin={margin:.3f} | SVs={len(sv)}",
                 fontsize=10)
    ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')

plt.suptitle('Effect of C on Linear SVM\n'
             'Small C = wide margin (regularized) | Large C = narrow margin (overfit risk)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observation: small C allows misclassifications to gain a wider, more general margin.")
print("Large C tries to classify everything correctly — even the noisy points near the boundary.")

<!-- Section 1.6 content moved to previous cell -->


In [ ]:
# ── Visualize linear vs. RBF kernel on non-linearly separable data ────────
datasets = {
    'Moons':   make_moons(n_samples=200, noise=0.15, random_state=42),
    'Circles': make_circles(n_samples=200, noise=0.10, factor=0.4, random_state=42),
}

kernels = {
    'Linear  SVM': SVC(kernel='linear', C=1.0,  random_state=42),
    'RBF     SVM': SVC(kernel='rbf',    C=1.0,  gamma='scale', random_state=42),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for row_idx, (ds_name, (X_ds, y_ds)) in enumerate(datasets.items()):
    for col_idx, (k_name, svm_model) in enumerate(kernels.items()):
        ax = axes[row_idx][col_idx]
        svm_model.fit(X_ds, y_ds)
        acc = svm_model.score(X_ds, y_ds)

        x0_min, x0_max = X_ds[:, 0].min() - 0.3, X_ds[:, 0].max() + 0.3
        x1_min, x1_max = X_ds[:, 1].min() - 0.3, X_ds[:, 1].max() + 0.3
        xx, yy = np.meshgrid(np.linspace(x0_min, x0_max, 300),
                             np.linspace(x1_min, x1_max, 300))
        Z = svm_model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

        ax.contourf(xx, yy, Z, alpha=0.3,
                    cmap=ListedColormap(['#d4e6f1', '#fadbd8']))
        ax.scatter(X_ds[y_ds == 0, 0], X_ds[y_ds == 0, 1],
                   c='steelblue', s=30, edgecolors='white', linewidths=0.5)
        ax.scatter(X_ds[y_ds == 1, 0], X_ds[y_ds == 1, 1],
                   c='tomato', s=30, edgecolors='white', linewidths=0.5)
        ax.set_title(f"{ds_name} | {k_name}\nTrain accuracy = {acc:.3f}", fontsize=10)
        ax.set_xlabel('x₁'); ax.set_ylabel('x₂')

plt.suptitle('The Kernel Trick: Linear vs. RBF SVM on Non-Linear Data',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("The linear kernel cannot separate moons or circles — no straight line will do.")
print("The RBF kernel implicitly maps data to a higher-dimensional space where separation IS linear.")

---

## **1.7 SVM Hyperparameters: C and γ**

### **1.7.1 The C–γ Interaction**

With the RBF kernel, model complexity is controlled jointly by two hyperparameters, and understanding their interaction is essential for tuning:

| | **Small γ** (wide influence, smooth) | **Large γ** (narrow influence, spiky) |
|--|--------------------------------------|----------------------------------------|
| **Small C** (forgiving) | Very smooth boundary — likely underfit | Moderate complexity; smooth but locally responsive |
| **Large C** (strict) | Moderate complexity; smooth but tight to data | Very complex, spiky boundary — likely overfit |

**Reading this table:** C controls how much the model is allowed to misclassify training points; γ controls the spatial reach of each support vector. Overfitting happens when the model is both unforgiving (large C) and myopic (large γ) — it memorizes every local detail of the training set. Underfitting happens when the model is both forgiving (small C) and globally influenced (small γ) — it can't form a boundary complex enough to separate the classes.

**Why they interact:** Imagine a dataset with genuine class overlap. Large γ creates many small, tight decision regions — each support vector claims a small territory. Large C forces the boundary to correctly classify every training point within those territories, even the noisy ones. The result is a fragmented, overfitted boundary. Now reduce C slightly: some noisy points are allowed to be misclassified, and the boundary smooths out. The lesson is that C and γ are not independent knobs — changing one affects how much the other matters.

**Practical strategy:** Grid search over a log-spaced grid, typically C ∈ {0.001, 0.01, 0.1, 1, 10, 100, 1000} and γ ∈ {0.0001, 0.001, 0.01, 0.1, 1, 10}. Use cross-validation with AUC-ROC or F1 as the scoring metric. Visualize the resulting grid as a heatmap (as in the code below) — it makes the interaction visible and helps identify whether you need to expand the search range.

### **1.7.2 Feature Scaling is Non-Negotiable**

SVMs are **extremely sensitive to feature scale**, and this is one of the most common mistakes practitioners make.

Here is why it matters so much for the RBF kernel specifically. The kernel computes:
$$K(\mathbf{x}_i, \mathbf{x}_j) = \exp\left(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2\right)$$

The squared Euclidean distance $\|\mathbf{x}_i - \mathbf{x}_j\|^2 = \sum_k (x_{ik} - x_{jk})^2$ is a sum over all features. If feature 1 ranges from 0 to 1 and feature 2 ranges from 0 to 10,000, then feature 2 contributes $(x_{i2} - x_{j2})^2$ values on the order of millions, while feature 1 contributes values on the order of 1. Feature 2 completely dominates the distance calculation — the kernel effectively ignores feature 1 entirely, no matter how informative it might be.

**StandardScaler** fixes this by transforming each feature to zero mean and unit variance:
$$x' = \frac{x - \mu}{\sigma}$$

After scaling, all features contribute equally to Euclidean distances. This is also why you should **always put the scaler inside the pipeline** — fitting the scaler on the training data and then applying it to the test data prevents data leakage. Fitting the scaler on the full dataset (before the train/test split) uses information from the test set to transform the training set, which is a subtle but real form of leakage.

> **Does Naïve Bayes need scaling?** No — and this is a key practical difference. Gaussian NB estimates a separate distribution for each feature independently. The scale of feature 2 affects only the estimated mean and variance of that feature, not its relationship to other features. The log-probabilities are computed per-feature and then summed, so a large-scale feature does not swamp a small-scale one. Multinomial and Bernoulli NB also don't require scaling — they operate on counts and binary values respectively.

### **1.7.3 How to Interpret a Trained SVM**

SVMs are often criticized as "black boxes," but this is somewhat unfair — especially for the linear kernel.

**Linear SVM:** The weight vector $\mathbf{w}$ has the same interpretation as logistic regression coefficients. Features with large positive $w_i$ push predictions toward the positive class; features with large negative $w_i$ push toward the negative class. You can rank features by $|w_i|$ to identify the most influential predictors.

**RBF SVM:** Interpretation is harder. You can't read off feature importances from $\mathbf{w}$ because the decision is a weighted combination of kernel evaluations, not a linear combination of original features. However, you can identify which training points are support vectors (they have non-zero dual coefficients) and examine what they look like — these are the most informative examples in your training set.

For truly interpretable models on non-linear problems, tree-based methods (which we'll cover in the coming weeks) are usually a better choice than kernel SVMs.


In [ ]:
# ── Effect of γ on decision boundary (fixed C) ────────────────────────────
X_rbf, y_rbf = make_moons(n_samples=200, noise=0.2, random_state=0)
X_rbf_tr, X_rbf_te, y_rbf_tr, y_rbf_te = train_test_split(
    X_rbf, y_rbf, test_size=0.3, random_state=42
)

gammas = [0.05, 1.0, 20.0]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, gamma in zip(axes, gammas):
    model = SVC(kernel='rbf', C=5.0, gamma=gamma, random_state=42)
    model.fit(X_rbf_tr, y_rbf_tr)
    tr_acc = model.score(X_rbf_tr, y_rbf_tr)
    te_acc = model.score(X_rbf_te, y_rbf_te)

    x0_min, x0_max = X_rbf[:, 0].min() - 0.3, X_rbf[:, 0].max() + 0.3
    x1_min, x1_max = X_rbf[:, 1].min() - 0.3, X_rbf[:, 1].max() + 0.3
    xx, yy = np.meshgrid(np.linspace(x0_min, x0_max, 300),
                         np.linspace(x1_min, x1_max, 300))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3,
                cmap=ListedColormap(['#d4e6f1', '#fadbd8']))
    ax.scatter(X_rbf_tr[y_rbf_tr == 0, 0], X_rbf_tr[y_rbf_tr == 0, 1],
               c='steelblue', s=35, edgecolors='white', linewidths=0.5, label='Class 0')
    ax.scatter(X_rbf_tr[y_rbf_tr == 1, 0], X_rbf_tr[y_rbf_tr == 1, 1],
               c='tomato', s=35, edgecolors='white', linewidths=0.5, label='Class 1')
    ax.set_title(f"γ = {gamma}\nTrain={tr_acc:.3f} | Test={te_acc:.3f}", fontsize=10)
    ax.set_xlabel('x₁'); ax.set_ylabel('x₂')

plt.suptitle('Effect of γ on RBF SVM (C = 5.0)\n'
             'Small γ = smooth, underfitting | Large γ = spiky, overfitting',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Grid search over C and γ: the canonical SVM tuning workflow ───────────
# Note: always scale BEFORE GridSearchCV in production — use a Pipeline.

from sklearn.pipeline import Pipeline

svm_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(kernel='rbf', probability=True, random_state=42))
])

param_grid = {
    'svm__C':     [0.01, 0.1, 1, 10, 100],
    'svm__gamma': [0.001, 0.01, 0.1, 1, 10],
}

X_gs, y_gs = make_classification(n_samples=600, n_features=10,
                                  n_informative=5, random_state=42)
X_gs_tr, X_gs_te, y_gs_tr, y_gs_te = train_test_split(
    X_gs, y_gs, test_size=0.25, random_state=42
)

gs = GridSearchCV(svm_pipe, param_grid, cv=5, scoring='roc_auc',
                  n_jobs=-1, refit=True, verbose=0)
gs.fit(X_gs_tr, y_gs_tr)

print(f"Best params  : {gs.best_params_}")
print(f"Best CV AUC  : {gs.best_score_:.4f}")
print()

# Evaluate the best model on held-out test set
best_model = gs.best_estimator_
y_proba_gs = best_model.predict_proba(X_gs_te)[:, 1]
y_pred_gs  = best_model.predict(X_gs_te)

print(f"Test Accuracy : {accuracy_score(y_gs_te, y_pred_gs):.4f}")
print(f"Test F1-Score : {f1_score(y_gs_te, y_pred_gs):.4f}")
print(f"Test AUC-ROC  : {roc_auc_score(y_gs_te, y_proba_gs):.4f}")

In [ ]:
# ── Heatmap of grid search scores (C vs γ) ───────────────────────────────
results_df = pd.DataFrame(gs.cv_results_)
scores = results_df['mean_test_score'].values.reshape(len(param_grid['svm__C']),
                                                       len(param_grid['svm__gamma']))

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(scores, interpolation='nearest', cmap='RdYlGn',
               vmin=scores.min(), vmax=1.0)
plt.colorbar(im, ax=ax, label='Mean CV AUC-ROC')

ax.set_xticks(range(len(param_grid['svm__gamma'])))
ax.set_xticklabels(param_grid['svm__gamma'])
ax.set_yticks(range(len(param_grid['svm__C'])))
ax.set_yticklabels(param_grid['svm__C'])
ax.set_xlabel('γ (gamma)')
ax.set_ylabel('C')
ax.set_title('Grid Search Heatmap: RBF SVM\nCell value = 5-fold CV AUC-ROC',
             fontsize=11)

for i in range(len(param_grid['svm__C'])):
    for j in range(len(param_grid['svm__gamma'])):
        ax.text(j, i, f"{scores[i, j]:.3f}", ha='center', va='center',
                fontsize=9, color='black')

plt.tight_layout()
plt.show()

print("Read the heatmap: bright green = best AUC. Red corners = poor choices.")
print("Pattern: large γ + large C → overfit. Small γ + small C → underfit.")

---

## **1.8 Algorithm Selection: When to Use What**

### **1.8.1 Head-to-Head Comparison**

Let's compare our three classifiers — Naïve Bayes, SVM, and Logistic Regression — on a single structured dataset using the full evaluation toolkit from Week 7.


In [ ]:
# ── Three-way comparison on an imbalanced structured dataset ─────────────
X_comp, y_comp = make_classification(
    n_samples=1000, n_features=15, n_informative=8,
    weights=[0.85, 0.15], random_state=42
)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_comp, y_comp, test_size=0.25, stratify=y_comp, random_state=42
)

# Scale once; all models share the same scaler for fairness
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

classifiers = {
    'Gaussian NB':          GaussianNB(),
    'SVM (RBF, balanced)':  SVC(kernel='rbf', C=10, gamma='scale',
                                class_weight='balanced', probability=True, random_state=42),
    'Logistic Reg (baseline)': LogisticRegression(class_weight='balanced',
                                                   max_iter=1000, random_state=42),
}

print(f"{'Model':<30} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>8} {'AUC':>8}")
print("─" * 78)

comp_results = {}
for name, clf in classifiers.items():
    clf.fit(X_tr_s, y_tr)
    preds  = clf.predict(X_te_s)
    probas = clf.predict_proba(X_te_s)[:, 1]

    acc  = accuracy_score(y_te, preds)
    pre  = precision_score(y_te, preds)
    rec  = recall_score(y_te, preds)
    f1   = f1_score(y_te, preds)
    auc  = roc_auc_score(y_te, probas)

    comp_results[name] = dict(acc=acc, pre=pre, rec=rec, f1=f1, auc=auc,
                               preds=preds, probas=probas)
    print(f"  {name:<28} {acc:>10.4f} {pre:>10.4f} {rec:>10.4f} {f1:>8.4f} {auc:>8.4f}")

In [ ]:
# ── ROC curves for all three models ──────────────────────────────────────
from sklearn.metrics import roc_curve

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ['steelblue', 'tomato', 'seagreen']
markers = ['o', 'D', 's']

for (name, res), color, marker in zip(comp_results.items(), colors, markers):
    fpr, tpr, _ = roc_curve(y_te, res['probas'])
    ax1.plot(fpr, tpr, color=color, lw=2,
             label=f"{name} (AUC={res['auc']:.3f})")

ax1.plot([0, 1], [0, 1], 'k--', lw=1)
ax1.set(xlabel='False Positive Rate', ylabel='True Positive Rate',
        title='ROC Curves — Three Classifiers')
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

# Bar chart of all metrics
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
x_pos = np.arange(len(metric_names))
width = 0.25

for idx, (name, res) in enumerate(comp_results.items()):
    vals = [res['acc'], res['pre'], res['rec'], res['f1'], res['auc']]
    ax2.bar(x_pos + idx * width, vals, width, label=name,
            color=colors[idx], alpha=0.8, edgecolor='white')

ax2.set_xticks(x_pos + width)
ax2.set_xticklabels(metric_names)
ax2.set_ylim(0, 1.05)
ax2.set_ylabel('Score')
ax2.set_title('Metric Comparison — Imbalanced Dataset (85/15)')
ax2.legend(fontsize=8)
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Three Classifiers, One Toolkit — The Week 7 Evaluation Framework in Action',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### **1.8.2 Decision Framework**

| Factor | Naïve Bayes | SVM | Logistic Regression |
|--------|-------------|-----|---------------------|
| **Training speed** | ✅ Very fast (single pass) | ⚠️ Slow on large datasets ($O(n^2)$–$O(n^3)$) | ✅ Fast |
| **Prediction speed** | ✅ Very fast | ✅ Fast (only support vectors needed) | ✅ Fast |
| **High-dimensional sparse data (text)** | ✅ Excellent | ✅ Good (linear kernel) | ✅ Good |
| **Small dataset** | ✅ Works well (few parameters) | ✅ Works well (max-margin is robust) | ✅ Works well |
| **Large dataset (n > 100k)** | ✅ Excellent | ❌ Can be prohibitively slow | ✅ Good |
| **Non-linear boundaries** | ❌ No | ✅ Yes, with kernel | ❌ No (without feature engineering) |
| **Probability calibration** | ⚠️ Often overconfident | ⚠️ Needs Platt scaling post-hoc | ✅ Well-calibrated natively |
| **Interpretability** | ✅ Feature log-probs (text) | ⚠️ Weight vector (linear only) | ✅ Coefficients + odds ratios |
| **Feature scaling required** | ❌ No | ✅ Yes — critical | ✅ Recommended |
| **Correlated features** | ❌ Hurts (double-counts evidence) | ✅ Handles well | ✅ Handles well |
| **Native multi-class support** | ✅ Yes | ⚠️ Requires OvR or OvO wrapper | ✅ Yes |
| **Missing features** | ✅ Handled gracefully | ❌ Requires imputation | ❌ Requires imputation |

### **1.8.3 The "Why" Behind Each Recommendation**

Rather than just listing when to use each algorithm, it's worth understanding *why* each recommendation holds — that understanding travels to new problems.

**Why Naïve Bayes for text?**  
Text features are approximately independent conditional on the class in many practical settings. Whether the word "free" appears is largely unrelated to whether the word "Nigeria" appears, given that we're already conditioning on spam. The naïve assumption is a much better approximation for text than for, say, tabular medical data where features like "height" and "weight" are deeply correlated. Additionally, text data is high-dimensional and sparse — exactly the regime where Naïve Bayes's parameter efficiency shines most.

**Why SVMs for small/medium non-linear problems?**  
The maximum-margin objective provides a strong inductive bias: it selects the boundary that is most "between" the classes, which generalizes well when training data is limited. Neural networks can learn non-linear boundaries too, but require far more data to avoid overfitting — they have many more parameters and no maximum-margin regularization. For datasets with a few hundred to a few thousand examples and non-linear structure, a tuned RBF-SVM often outperforms both simpler and more complex models.

**Why Logistic Regression as a baseline?**  
Logistic regression is the simplest model that produces well-calibrated probabilities, has interpretable coefficients, scales to large datasets with stochastic gradient descent, and handles missing data with imputation. These properties make it an ideal first model on almost any tabular classification problem. If logistic regression already achieves 95% of the performance of the best model you can find, the added complexity of SVMs or tree ensembles may not be worth the interpretability cost.

**Why does training speed matter so much for SVMs?**  
The standard SVM solver (Sequential Minimal Optimization, or SMO) has time complexity $O(n^2)$ to $O(n^3)$ in the number of training examples. With 1,000 examples, this is fine. With 100,000 examples, it may take hours. With 10 million examples, it is essentially intractable. Naïve Bayes and logistic regression both scale linearly — they remain practical at any dataset size. This is a genuine, hard constraint, not a theoretical concern.

### **1.8.4 Practical Starting Points**

**Use Naïve Bayes when:**
- Your features are text (word counts or word presence/absence)
- You need a fast, interpretable baseline with very little tuning
- Your dataset is very large and training speed is a constraint
- Features are approximately independent given the class
- You need to handle missing features gracefully at prediction time

**Use SVM when:**
- Your dataset is small to medium (< ~50k samples)
- You need a non-linear boundary and don't have enough data for deep learning
- Your features are high-dimensional and sparse (text with linear kernel)
- You want the theoretical guarantees of maximum-margin classification
- Precise probability estimates are not required

**Use Logistic Regression when:**
- You need well-calibrated probability outputs (risk scoring, threshold tuning)
- You need a fast, interpretable baseline to understand feature importances
- Your data is large and scalability matters
- You want to combine with other models (calibrated probabilities stack well)
- You're presenting results to non-technical stakeholders who need explanations

> **The honest answer:** On most real-world tabular classification problems, logistic regression and gradient boosted trees (which we'll cover soon) dominate. SVMs are powerful but operationally awkward — they require careful scaling, slow training, and limited interpretability with non-linear kernels. Naïve Bayes is a specialist tool that excels in its niche. Knowing all three gives you the ability to choose the right tool for the right situation — and to explain *why* to someone who asks.


---

## **1.9 Faith Integration**

> *"For the Lord gives wisdom; from his mouth come knowledge and understanding."* — Proverbs 2:6

Today we saw two very different approaches to the same fundamental problem:

**Naïve Bayes** begins with prior belief and updates it with evidence. This is, at its core, a model of rational learning — how a reasonable mind should revise its beliefs when new information arrives. The prior matters: a well-informed prior leads to better conclusions from the same evidence. This mirrors a principle we know from faith: what we believe before examining the evidence shapes how we interpret it. Epistemic humility — holding our priors loosely — is as important in data science as it is in wisdom.

**SVMs** take a different approach: they find the widest possible road between competing classes, and they do so by focusing entirely on the hardest, most ambiguous cases — the support vectors. The margin is the buffer of grace — the space where we acknowledge uncertainty. A model that hugs the training data too tightly (small margin, large C) fails to generalize because it has no room for grace, no acknowledgment that new data might not conform exactly to what it has seen before.

Both models remind us that **wisdom involves knowing your own limits.** The naïve independence assumption is wrong — but it's a known, manageable wrong. The kernel trick acknowledges that the world is not always linearly separable — and finds a way to work with that complexity without being overwhelmed by it. In our own reasoning, we do well to know where our mental models simplify too aggressively, and where we need richer frameworks.

---

## **BREAK (10-15 minutes)**

---

## **2.1 Lab Exercises** (new notebook)

---

## **3.1: Review & Wrap-Up**

### **3.1.1 Key Takeaways**

1. **Bayes' theorem updates prior belief with evidence** — the posterior $P(C \mid \mathbf{x})$ is proportional to likelihood times prior; the denominator $P(\mathbf{x})$ cancels because it's the same for all classes

2. **The naïve independence assumption** makes joint likelihood tractable by factoring it into a product of per-feature likelihoods — wrong in theory, effective in practice

3. **Three NB variants for three feature types** — Gaussian NB for continuous features (models each feature's distribution as a normal); Multinomial NB for word counts; Bernoulli NB for binary word presence

4. **Laplace smoothing** adds a small count to every feature/class combination to prevent zero-probability estimates from dominating the posterior

5. **Log-probabilities replace products** — summing log-likelihoods avoids floating-point underflow when many small probabilities are multiplied together

6. **Naïve Bayes is a generative model** — it models how each class generates features; logistic regression is discriminative — it models the boundary directly

7. **Naïve Bayes shines on text classification** — fast, interpretable, data-efficient, and competitive with logistic regression; most informative words per class are directly inspectable

8. **SVMs find the maximum-margin hyperplane** — among all hyperplanes that correctly classify training data, choose the one that maximizes the perpendicular distance to the nearest points of each class

9. **Support vectors are the only training points that matter** — remove any other training example and the boundary stays the same; this makes SVMs robust and memory-efficient at inference time

10. **C controls the bias-variance tradeoff** — small C allows margin violations (regularization, wider margin, better generalization); large C penalizes violations heavily (narrow margin, overfit risk)

11. **The kernel trick computes inner products in a high-dimensional space implicitly** — never constructing the expanded feature space, allowing non-linear boundaries at the cost of only evaluating $K(\mathbf{x}_i, \mathbf{x}_j)$

12. **RBF kernel is the default non-linear choice** — similarity between points decays exponentially with Euclidean distance; $\gamma$ controls the decay rate

13. **Large $\gamma$ → spiky overfit boundary; small $\gamma$ → smooth underfit boundary** — grid search over a log-spaced C×γ grid is the canonical tuning approach

14. **Feature scaling is mandatory for SVMs** — the RBF kernel uses Euclidean distance; unscaled features with large ranges dominate the distance computation and effectively ignore all others

15. **SVMs do not natively produce probabilities** — `probability=True` in sklearn uses Platt scaling (a post-hoc logistic regression on the SVM scores), which is slower and adds slight inaccuracy

16. **Algorithm selection is a design decision** — consider training speed, dataset size, linearity of the boundary, probability calibration needs, interpretability requirements, and feature type

17. **The Week 7 evaluation toolkit applies to all classifiers** — precision, recall, F1, AUC-ROC, PR-AUC, and the performance diagram work identically regardless of which algorithm produced the predictions



---

## **3.2 Coming Up**

### **3.2.1 Next Week (Week 9): Decision Trees & Feature Engineering**

1. **Decision Tree Classifiers**
   - How trees split: Gini impurity and information gain
   - Visualizing tree structure and the splits it learns
   - Hyperparameter tuning: max depth, min samples per leaf

2. **Feature Engineering**
   - Creating polynomial and interaction features
   - How new features change what a model can learn
   - Measuring feature importance from decision trees

3. **Overfitting Deep Dive**
   - Why unpruned decision trees memorize training data
   - Learning curves as a diagnostic tool
   - How Week 5's cross-validation framework applies here

### **3.2.2 Why This Matters**

Decision trees are the building blocks of the two most powerful ensembles in applied machine learning — Random Forests (Week 10) and Gradient Boosting (Week 11). Understanding trees deeply now will make those weeks significantly easier. Feature engineering also re-appears in the final project: your choice of features is often more impactful than your choice of algorithm.

**Prepare by:**
- Completing this week's lab on the SMS Spam / structured dataset
- Reviewing the cross-validation material from Week 5 (K-fold, GridSearchCV)
- Thinking about your final project dataset — which of today's algorithms might apply?

### **3.2.3 Reminders**
- **Lab due:** Monday, 30 March @ 6:00 PM (grace period: Wednesday, 1 April @ 11:59 PM)
- **Project Check-in:** Due tonight! Submit your one-page proposal by 11:59 PM
- **Office hours this week:** Monday 4:30–5:50 PM (in-person) and Wednesday 4:30–5:50 PM